<a id="unit-u.3fb2fa1fd4e6467a89b1465918087ea1"></a>

<div>
  <div>04-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Талдамалық жүктемелерге деректерді модельдеу</div>
  <p>Дереккөз шарты мен талдамалық талаптарды айқын түйіршіктілігі және кесте моделі бар типтелген Doris кестелеріне айналдырыңыз.</p>
  <span>Doris 4.1.3 · Түйіршіктілік · Кесте модельдері · Типтелген шекаралар · Auto Range Partition · Hash Bucket</span>
</div>

**4-модуль: Apache Doris жүйесінде деректерді модельдеу** · [Сабақ](course.md) · [Тест](quiz04_schema_and_modeling_choices.ipynb)

Шамамен 30 минут жоспарлаңыз.

Бұл зертханалық жұмыстың соңында дереккөз және есеп беру талаптарын екі айқын түйіршіктілікке айналдырып, Doris дерек типтері мен жетіспейтін мән ережелерін таңдап, әр кесте жауап беруі тиіс сұрақтардан егжей-тегжейлі дерек моделін және күндік жиынтық моделін шығарасыз. Ұяшықтарды ретімен орындаңыз.

<a id="unit-u.110407893f5e4115ba94f25154b2bc38"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

Келесі бастапқы баптау ұяшығын 1-бөлімге дейін орындаңыз. Ол ортақ курс көмекшісін жүктейді, 1-деңгейде қолданылған нәтиже стильдерін орнатады және кейінгі әр код ұяшығы пайдаланатын `lab` нысанын жасайды.

Jupyter ядросын қайта іске қосқан сайын оны қайта орындаңыз. Ол Docker іске **қоспайды**, кесте жасамайды және деректерді өзгертпейді. Кейінгі ұяшықтар бар Doris сынақ ортасына қайта қосыла алады. Жасыл **Lab tools are ready** хабары шықпайынша жалғастырмаңыз.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


<a id="unit-u.67aeb19971284d00b6a1e0463779f885"></a>

<a id="1-inspect-the-source-column-contract"></a>

## 1. Бастапқы баған шартын тексеру

4-модуль 1-деңгейдің `events` кестесінен жаңа модельдер құрады. Бірінші ұяшық сынақ ортасы тоқтап тұрса оны іске қосып, толық дереккөздің қолжетімді екенін растайды; содан кейін әр өрістің бар типін, NULL рұқсатын және әдепкі мәнін тексеріңіз.

Дереккөз шарты идентификаторлар сандық екенін, `event_time` уақыт талдауын қолдайтынын, `revenue` дәл ондық арифметиканы қажет ететінін және `event_type` пен `region` шектелген санаттар екенін айтады. Тұрақты баған типтерін осы файлдағы ең үлкен мән емес, сол талаптар анықтайды. 4-бөлім кесте моделін таңдардың дәл алдында жол түйіршіктілігін тексереді.

In [ ]:
lab.start_container()
lab.connect()
lab.execute("USE doris_course")
lab.check_prerequisites(4)

lab.sql("""
SELECT COUNT(*) AS source_rows
FROM events
""", title="Source availability");

<a id="unit-u.14ccac68e51c4d47a39b248e5fb88284"></a>

**Күтілетін нәтиже**

Жасыл белгі 1-зертханалық жұмыстың `events` кестесінде 10,158,080 жол барын растайды; ол жоқ немесе толық емес болса, ұяшық тоқтап, алдымен қай зертханалық жұмысты аяқтау керектігін атайды.

| source_rows |
|---:|
| 10,158,080 |

`source_rows` толық 1-деңгей деректер жиыны барын растайды. Бұл тексеру тек дереккөз қолжетімділігін анықтайды; ол бір жолдың мағынасын әлі анықтамайды.

<a id="unit-u.2039db85f14a41c69cada26d61b0d8d9"></a>

<a id="inspect-the-selected-column-contract"></a>

### Таңдалған баған шартын тексеру

Метадерек сұрауы әр өрістің бар Doris типін, NULL рұқсатын және әдепкі мәнін көрсетеді. Оны шарт ретінде оқыңыз: әр таңдау тек ағымдағы үлгі мәндерінен көшірілмей, дереккөз мағынасымен немесе талдамалық амалмен негізделуі тиіс.

In [ ]:
lab.sql(
    "SHOW FULL COLUMNS FROM events",
    title="Event-detail column contract",
    columns=["Field", "Type", "Null", "Default"],
);


<a id="unit-u.77b139ff123644cfbf000c57ef5b2846"></a>

**Күтілетін нәтиже**

Метадерек нәтижесі мына шартты көрсетуі тиіс:

| Өріс тобы | Таңдалған шарт | Оны негіздейтін талап |
|---|---|---|
| `event_id`, `user_id`, `product_id` | `BIGINT NOT NULL` | Сандық идентификаторлар жоғары тұрған шартқа сай және өсу қорын сақтайды. |
| `event_time` | `DATETIME NOT NULL` | Сұраулар оқиғаларды уақыт бойынша сүзеді, топтайды және реттейді. |
| `event_type` | `VARCHAR(32) NOT NULL` | Шектелген санатқа бақыланатын болашақ мәндерге орын қажет. |
| `region` | `VARCHAR(16) NOT NULL` | Қысқа, шектелген санатқа шексіз жол қажет емес. |
| `revenue` | `DECIMAL(12,2) NOT NULL` | Валютаға дәл бекітілген нүктелі арифметика қажет. |

Нәтиже сақталған схеманың таңдалған шартқа сәйкес екенін тексереді. Атап айтқанда, `DECIMAL(12,2)` дәл валюталық арифметиканы қолдайды, ал шектелген `VARCHAR` бағандары бақыланатын санаттарды көрсетеді. Тип дереккөз шартын және күтілетін талдамалық амалдарды қамтуы тиіс; оны тек бүгін байқалған ең үлкен мәнді көшіру арқылы таңдауға болмайды.

<a id="unit-u.7a7a8b31dc334de3bd29ef2caab8f3ca"></a>

<a id="2-observe-the-cost-of-a-permissive-source-schema"></a>

## 2. Еркін бастапқы схеманың шығынын бақылау

Бастапқы қабылдау кестесі мағынасы тексерілмей тұрып бастапқы мәндерді сақтау үшін `VARCHAR` пайдалана алады. Бұл икемділік сандық идентификаторлардың, уақыт белгілерінің және түсімнің Doris ішіне мәтін ретінде кіруіне де мүмкіндік береді, сондықтан схема олардың күтілетін амалдарды қолдайтынын әлі кепілдей алмайды.

Үлгіде бес бастапқы жол бар. Төрт түсім мәні жарамды ондық мәтін; біреуі `not-a-number`. `region_text` және `product_id_text` NULL рұқсат етеді, бірақ жетіспейтін мәндердің бизнес мағынасы әртүрлі: оқиғаға пайдалануға болатын аймақ әлі қажет, ал өнімнің болмауына рұқсат етілген.

Бірінші нәтиже түсім мәтін болып қалғанда қажет жұмысты көрсетеді. Талдамалық сұрауда `TRY_CAST` қайталануы тиіс және сәтсіз түрлендірулер бөлек саналуы керек, өйткені `SUM` пайда болған `NULL` елемейді. Екінші нәтиже жуық `DOUBLE` арифметикасы мен дәл `DECIMAL` арифметикасының айырмасын көрсету үшін әдейі үлкен сома пайдаланады.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS modeling_raw_events (
    event_time_text VARCHAR(32) NOT NULL,
    event_id_text VARCHAR(32) NOT NULL,
    user_id_text VARCHAR(32) NOT NULL,
    event_type_text VARCHAR(32) NOT NULL,
    region_text VARCHAR(16) NULL,
    product_id_text VARCHAR(32) NULL,
    revenue_text VARCHAR(32) NOT NULL
)
DUPLICATE KEY(event_time_text, event_id_text)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE modeling_raw_events")

lab.insert("""
INSERT INTO modeling_raw_events VALUES
    ('2020-03-01 09:00:00', '90001', '7001', 'view',     'region_01', '1005115', '0.00'),
    ('2020-03-01 09:01:00', '90002', '7001', 'cart',     'region_01', '1005115', '0.00'),
    ('2020-03-01 09:02:00', '90003', '7001', 'purchase', 'region_01', '1005115', '29.95'),
    ('2020-03-01 09:03:00', '90004', '7002', 'purchase', 'region_03', '13200021', 'not-a-number'),
    ('2020-03-01 09:04:00', '90005', '7003', 'purchase', NULL,        NULL,       '9.99')
""", title="Write the permissive source sample")

lab.sql("""
SELECT
    COUNT(*) AS raw_rows,
    SUM(TRY_CAST(revenue_text AS DECIMAL(12,2))) AS calculated_revenue,
    SUM(CASE
        WHEN TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NULL THEN 1
        ELSE 0
    END) AS invalid_revenue_rows
FROM modeling_raw_events
""", title="Revenue calculated from text")

lab.sql("""
SELECT
    CAST('90071992547409.92' AS DOUBLE) AS float_amount,
    CAST('90071992547409.92' AS DOUBLE) + 0.01 AS float_plus_one_cent,
    CAST('90071992547409.92' AS DECIMAL(18,2)) + 0.01 AS decimal_plus_one_cent
""", title="Approximate and exact amount arithmetic");


<a id="unit-u.98238504e3c048d0970d024d0b0e1c70"></a>

**Күтілетін нәтиже**

Бастапқы кесте `not-a-number` қоса барлық бес жолды қабылдайды, өйткені `VARCHAR` мәннің тек мәтін екенін тексереді. Ол түсімнің бизнес мағынасын тексермейді.

| raw_rows | calculated_revenue | invalid_revenue_rows |
|---:|---:|---:|
| 5 | 39.94 | 1 |

Жалғыз `calculated_revenue` — толық емес дәлел: `TRY_CAST` қате мәнді `NULL` етіп түрлендіреді, ал `SUM` оны елемейді. Сондықтан осы мәтін бағаны талданатын барлық жерде бөлек жарамсыз жол саны қажет.

| float_amount | float_plus_one_cent | decimal_plus_one_cent |
|---:|---:|---:|
| 90071992547409.92 | 90071992547409.94 | 90071992547409.93 |

Осы шамада жуық `DOUBLE` мәніне бір цент қосу екі цент өзгеріс береді, ал `DECIMAL(18,2)` дәл ақшалай нәтиже береді. Үлкен мән тек екілік жылжымалы нүкте дәлдігін көрсету үшін пайдаланылады; оқиға моделі түсім ауқымына қажетті кішірек `DECIMAL(12,2)` шартын қолдануды жалғастырады.

Бастапқы схема әр талдамалық оқиғаға пайдалануға болатын аймақ қажет болса да, `region_text` үшін NULL рұқсат етеді. Бұл ереже шешілмесе, кейінгі сұраулар жетіспейтін аймақты қабылдамау, `NULL` ретінде топтау немесе ауыстыру керектігін қайта-қайта шешуге мәжбүр болар еді.

<a id="unit-u.1705693267d74f84bc3aba74575e489b"></a>

<a id="3-establish-the-typed-contract-at-ingestion"></a>

## 3. Деректерді қабылдауда типтелген шартты орнату

Ішкі кесте әр бағанға бір бизнес мағынасын тағайындайды: идентификаторлар `BIGINT`, уақыт `DATETIME`, түсім дәл `DECIMAL(12,2)` пайдаланады. Қысқа санат өрістері жалпылама тым үлкен жолдың орнына шектелген `VARCHAR` ұзындықтарын пайдаланады. Бір жол әлі де бір бастапқы оқиғаны білдіргендіктен, кесте Duplicate Key моделін пайдаланады.

Түрлендіру мен тексеру енді жолдар еркін дереккөзден типтелген кестеге өткенде бір рет орындалады. `TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NOT NULL` қате түсім жолын алып тастайды; кейінгі талдамалық сұраулар сол түрлендіруді қайталамайды.

`DEFAULT` пен `NULL` әртүрлі мағына беретіндіктен, екі енгізу қажет:

1. Аймағы бар жолдар `region_text` мәнін `region` бағанына айқын жазады.
2. Аймағы жоқ жарамды жол мақсатты бағандар тізімінен `region` өрісін түсіріп қалдырады. Сондықтан Doris `DEFAULT "unknown"` береді. `NULL` айқын жазылса, `region NOT NULL` бұзылар еді; әдепкі мән берілген мән `NULL` болған сайын емес, баған түсірілгенде қолданылады.

Өнімсіз оқиға осы шартта жарамды болғандықтан, `product_id` NULL рұқсат етілетін болып қалады. Осылайша схема `region` үшін қажетсіз NULL рұқсатын алып тастап, барлық жетіспейтін мәнді қатеге айналдырмайды.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS modeling_typed_events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL DEFAULT "unknown",
    product_id BIGINT NULL,
    revenue DECIMAL(12,2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE modeling_typed_events")

lab.insert("""
INSERT INTO modeling_typed_events (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    CAST(event_time_text AS DATETIME),
    CAST(event_id_text AS BIGINT),
    CAST(user_id_text AS BIGINT),
    event_type_text,
    region_text,
    CAST(product_id_text AS BIGINT),
    TRY_CAST(revenue_text AS DECIMAL(12,2))
FROM modeling_raw_events
WHERE region_text IS NOT NULL
  AND TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NOT NULL
""", title="Write valid rows with an explicit region")

lab.insert("""
INSERT INTO modeling_typed_events (
    event_time, event_id, user_id, event_type, product_id, revenue
)
SELECT
    CAST(event_time_text AS DATETIME),
    CAST(event_id_text AS BIGINT),
    CAST(user_id_text AS BIGINT),
    event_type_text,
    CAST(product_id_text AS BIGINT),
    TRY_CAST(revenue_text AS DECIMAL(12,2))
FROM modeling_raw_events
WHERE region_text IS NULL
  AND TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NOT NULL
""", title="Omit region to apply its default")

lab.sql(
    "SHOW FULL COLUMNS FROM modeling_typed_events",
    title="Typed column contract",
    columns=["Field", "Type", "Null", "Key", "Default"],
)

lab.sql("""
SELECT
    COUNT(*) AS typed_rows,
    SUM(revenue) AS typed_revenue,
    SUM(CASE WHEN region = 'unknown' THEN 1 ELSE 0 END) AS defaulted_regions,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_ids
FROM modeling_typed_events
""", title="Typed-table calculation");


<a id="unit-u.958bd74aadf144f8a15df89e71063734"></a>

**Күтілетін нәтиже**

Метадерек нәтижесі шекараны көрінетін етуі тиіс:

| Бастапқы көрініс | Типтелген ішкі кесте шарты | Қабылдаудан кейінгі әсер |
|---|---|---|
| Жалпы мәтін бағандарында сақталған ID мәндері | `BIGINT NOT NULL` | Сандық сәйкестікке орындалу кезіндегі түрлендіру енді қажет емес |
| Мәтін түрінде сақталған уақыт белгісі | `DATETIME NOT NULL` | Уақыт бойынша сүзу мен реттеу уақыт типін пайдаланады |
| Мәтін түрінде сақталған түсім | `DECIMAL(12,2) NOT NULL` | Ақшалай агрегаттау дәл бекітілген нүктелі мәндерді пайдаланады |
| `region_text` `NULL` рұқсат етеді | `region NOT NULL DEFAULT "unknown"` | Әр сақталған оқиғаның пайдалануға болатын аймақ санаты бар |
| `product_id_text` `NULL` рұқсат етеді | `product_id BIGINT NULL` | Өнім идентификаторының болмауы жарамды әрі ажыратылатын қалады |

Типтелген кесте есебі мынаны көрсетуі тиіс:

| typed_rows | typed_revenue | defaulted_regions | null_product_ids |
|---:|---:|---:|---:|
| 4 | 39.94 | 1 | 1 |

Сандар ауысуды салыстырады: бес бастапқы жол минус бір қате түсім жолы — төрт типтелген жол. Сол жолдардың бірі аймақ әдепкісін алады, ал оның міндетті емес өнімі `NULL` болып қалады.

Талдамалық өрнек енді жай `SUM(revenue)`. 2-қадамға `SUM(TRY_CAST(revenue_text AS DECIMAL(12,2)))` және жарамсыз жол саны қажет болды, өйткені ол тексеруді сұрау уақытына қалдырды. Типтелген кесте бұл жұмысты қабылдау кезінде бір рет орындап, кейінгі сұрауларға тұрақты шарт береді.

<a id="unit-u.8484a5a986004332849dbc208c531d3f"></a>

<a id="4-build-the-complete-query-ready-event-model"></a>

## 4. Толық, сұрауға дайын оқиға моделін құру

Бес жолдық мысал тип түрлендіруін және екі ықтимал жетіспейтін мән саясатын көрсетті. Толық `events` кірісі қатаңырақ шартқа сай: әр жол аймақ пен өнім идентификаторын береді. Сондықтан соңғы `events_modelled` кестесі `region` бағанын үлгідегі айқын `unknown` әдепкі мәнінсіз `NOT NULL` күйінде ұстайды және `product_id` бағанын NULL рұқсат етілетін емес, `NOT NULL` етеді. Ол типтелген бағандарды қайта пайдаланып, осы толық кірістен сол екі өрісті талап етеді; `modeling_typed_events` ішіндегі барлық жетіспейтін мән ережесін көшірмейді.

Бір жол әлі бір бастапқы оқиғаны білдіреді, сондықтан соңғы модель барлық 10,158,080 жолда егжей-тегжейлі оқиға түйіршіктілігін сақтайды. Түйіршіктілік — жолдың бизнес мағынасы; ол бүкіл кесте бойынша қымбат бірегей жиын құрумен шығарылмайды. Енгізуден кейінгі салыстырып тексеру толық оқиғалар жиынының көшірілгенін растау үшін жол саны мен түсімді салыстырады.

Модель оқиға жүктемесінен шығарылған физикалық құрылым таңдауларын да қосады:

| Жұмыс жүктемесі талабы | Кесте шешімі |
|---|---|
| Әр бастапқы оқиғаны сақтау | Duplicate Key моделі |
| Оқиға тарихын уақытпен сүзу және деректерді күнтізбелік кезеңмен сақтау | `event_time` бойынша айлық Auto Range Partition |
| Уақыт ауқымы сканерлеулеріне реттелген сақтауды пайдалы ету | Сұрыптау кілті `event_time` бағанынан басталады |
| Пайдаланушы оқиғаларын болжамды тарату және параллель сканерлеуді қолдау | Әр Partition ішінде 10 Bucket бар `HASH(user_id)` |
| Бір BE бар курс сынақ ортасында орындау | Бір реплика |

`events_modelled` егжей-тегжейлі оқиға түйіршіктілігінде қалады: бір жол әлі бір бастапқы оқиғаны білдіреді. Қайталанатын Key мәндері ауыстыруды немесе агрегаттауды емес, оқиға тарихын білдіретіндіктен, кесте Duplicate Key моделін пайдаланады. Partition, Bucket және сұрыптау кілті физикалық ұйымдастыруды өзгертеді; жол мағынасын өзгертпейді және дерек жоймайды. Толық енгізуден бұрын бар мақсатты кесте тазартылады, сондықтан ұяшықты қайта орындау сол кесте мазмұнын береді.

Модельді құрып, салыстырып тексергеннен кейін жасалған метадеректерді тексеріңіз. DDL саясатты айтады; `SHOW PARTITIONS` және `SHOW TABLETS` осы ортада жүктелген мәндер үшін Doris нақты жасаған орналасуды көрсетеді.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS events_modelled (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id, user_id)
AUTO PARTITION BY RANGE (date_trunc(event_time, 'month'))
()
DISTRIBUTED BY HASH(user_id) BUCKETS 10
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE events_modelled")

lab.insert("""
INSERT INTO events_modelled (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    event_time, event_id, user_id, event_type, region, product_id, revenue
FROM events
""", title="Build the complete query-ready event model")

lab.sql("""
SELECT
    'events' AS table_name,
    COUNT(*) AS row_count,
    SUM(revenue) AS total_revenue
FROM events
UNION ALL
SELECT
    'events_modelled',
    COUNT(*),
    SUM(revenue)
FROM events_modelled
ORDER BY table_name
""", title="Source and final-model reconciliation");

<a id="unit-u.2bf2a15f6d5f4e81ad5096fc27192825"></a>

**Күтілетін нәтиже**

Дереккөз бен соңғы модель бір нәтижеде сәйкес келуі тиіс:

| table_name | row_count | total_revenue |
|---|---:|---:|
| events | 10,158,080 | 39,984,455.64 |
| events_modelled | 10,158,080 | 39,984,455.64 |

Дереккөз шарты бір жолдың бір бастапқы оқиғаны білдіретінін анықтайды. Екі кестедегі жол саны мен түсімнің сәйкес келуі толық енгізудің оқиға деректерін агрегаттамағанын немесе түсіріп қалдырмағанын тексереді. `events_modelled` 5, 6, 8 және 10-модульдер үшін сұрауға дайын егжей-тегжейлі оқиға кестесі ретінде қолжетімді болып қалады.

<a id="unit-u.9d32e45deaa547da980d3ce4df99833d"></a>

<a id="inspect-the-generated-partition-and-tablet-layout"></a>

### Жасалған Partition және Tablet орналасуын тексеру

Жариялау айлық Auto Range Partitioning және әр Partition ішінде он Hash Bucket пайдаланады. Кіріс мәнге қажет болғанда Auto Partitioning сәйкес ауқымды жасайды; ол ең кіші және ең үлкен уақыт белгілерінің арасындағы әр айды алдын ала жасамайды.

`SHOW PARTITIONS` және `SHOW TABLETS` әр BE шамамен минутына бір рет FE түйініне хабарлайтын жол сандарын, өлшемдер мен нұсқа сандарын көрсетеді. Жүктеуден немесе қайта іске қосудан кейін бірден олар әлі `0` немесе `-1` көрсетеді, сондықтан келесі ұяшық алдымен Tablet нысандары кестенің барлық жолы туралы хабарлағанша күтеді. Бұл шамамен екі минутқа дейін созылуы мүмкін.

Екі деңгейді бөлек тексеріңіз:

- `SHOW PARTITIONS` логикалық уақыт ауқымдарын, тарату кілтін және анықталған Bucket санын көрсетеді.
- `SHOW TABLETS` сол Partition және Bucket таңдауларынан жасалған физикалық шардтарды көрсетеді. Көмекші құрал әр реплика жазбасын шығармай-ақ байланыс көрінуі үшін Tablet жолдарын қорытындылайды.

In [ ]:
lab.wait_for_tablet_rows("events_modelled", expected_rows=10_158_080)

lab.sql(
    "SHOW PARTITIONS FROM events_modelled ORDER BY PartitionName",
    title="Generated monthly Partitions",
    columns=[
        "PartitionName", "State", "Range", "DistributionKey",
        "Buckets", "ReplicationNum", "RowCount",
    ],
)

lab.sql(
    "SHOW TABLETS FROM events_modelled",
    title="Generated Tablet layout",
    metadata_view="tablets",
);

<a id="unit-u.7b028b00f1344378a5053ac6028b07d5"></a>

**Күтілетін нәтиже**

Жасыл белгі `events_modelled` кестесінің 20 Tablet нысаны мен екі Partition бөлімі барлық 10,158,080 жол туралы хабарлағанын көрсетеді. Одан кейін:

`SHOW PARTITIONS` бастапқы деректерде нақты кездесетін айлар үшін екі `NORMAL` Partition қайтарады:

| Көрсетілетін бастапқы ай | Жартылай ашық Partition ауқымы | Тарату кілті | Bucket саны | Логикалық жолдар |
|---|---|---|---:|---:|
| 2019 жылғы желтоқсан | `[2019-12-01, 2020-01-01)` | `user_id` | 10 | 4,768,620 |
| 2020 жылғы наурыз | `[2020-03-01, 2020-04-01)` | `user_id` | 10 | 5,389,460 |

Қаңтар мен ақпан Partition бөлімдері жоқ, өйткені ешбір жүктелген оқиға сол ауқымдарды қажет етпейді. Бұл — ағымдағы бастапқы мәндер үшін Auto Partitioning бақыланған әсері; Doris әрдайым екі Partition жасайды деген мәлімдеме емес.

Tablet қорытындысы **20 бірегей Tablet** көрсетеді: жасалған екі Partition көбейтілген әр Partition ішіндегі он Bucket. Сынақ ортасында әр Tablet-ке бір реплика бар, өйткені кесте айқын бір реплика пайдаланады және ортада бір Backend (BE) бар. `Rows per tablet` бірдей бір мәннің орнына ауқым береді; Hash тарату мәндерді `user_id` хэшіне сай таратуды көздейді, бірақ дәл тең жол санын уәде етпейді.

Бұл метадерек нәтижелері жасалған иерархия мен таратуды көрсетеді. Олар Partition іріктеуін, көп түйінді параллель жылдамдықты, Rowset не Segment сияқты сақтау ішкі құрылымдарын немесе балама Bucket таңдауларының өнімділік айырмасын өлшемейді.

<a id="unit-u.fb5e751f3b324226b978a8d98160a5d4"></a>

<a id="5-translate-a-reporting-requirement-into-a-summary-model"></a>

## 5. Есеп беру талабын жиынтық моделіне айналдыру

Есеп беру талабы: **әр есеп үшін жеке оқиғаларды сканерлемей, аймақ және оқиға түрі бойынша күндік оқиға санын және түсімді қайтару**. Key синтаксисінен бастаудың орнына осы сұрақтан бастап, содан кейін кестені шығарыңыз.

| Есеп беру талабы | Модельдеу шешімі |
|---|---|
| Есептерді күн, аймақ және оқиға түрі бойынша топтау | Бір жол бір `(event_date, region, event_type)` тіркесін білдіреді. |
| Оқиғаларды санау және түсімді қосу | Қосылатын `event_count` және `total_revenue` метрикаларын `SUM` арқылы сақтау. |
| Бұл жиынтықтан жеке оқиға және пайдаланушы идентификаторлары сұралмайды | `event_id`, `user_id` немесе `product_id` өрістерін осы түйіршіктілікке қоспау. |
| Кейінгі топтамаларда бар күн-аймақ-түр тіркесі болуы мүмкін | Қайталанатын тіркестер жарияланған агрегаттау функциялары арқылы бірігуі үшін Aggregate Key моделін пайдалану. |

Түйіршіктілік пен Key бағандары байланысты, бірақ бір ұғым емес. Түйіршіктілік — бір жолдың бизнес мағынасы. Key бағандары сол мағынаны кесте анықтамасында іске асырады. Мұнда `event_date`, `region` және `event_type` Aggregate Key болады, өйткені олар бірге таңдалған есеп түйіршіктілігіндегі бір жолды анықтайды.

Бұл бұрын жасалған егжей-тегжейлі оқиға моделінен өзгеше. Ол әр жарамды бастапқы оқиға қолжетімді қалуы тиіс болғандықтан Duplicate Key пайдаланады. Ағымдағы күй моделі бір бизнес кілт тек соңғы жолын көрсетуі тиіс болғанда Unique Key пайдаланар еді. Таңдау Key бағандарының санына емес, жол нені білдіретініне және қайталанатын кілт мағынасына байланысты.

| Қайталанатын кілт талабы | Кесте моделі | Key бағандарының міндеті |
|---|---|---|
| Сол Key мәндері қайта келгенде әр бастапқы жолды сақтау | Duplicate Key | Сұрыптау кілтін анықтау; қайталанатын кілт жолдары көрінетін қалады |
| Әр бизнес кілтке бір ағымдағы жол көрсету | Unique Key | Сұрыптау кілтін және upsert үшін қолданылатын логикалық бірегейлікті анықтау |
| Сол топтау өлшемдері қайта келгенде метрикаларды біріктіру | Aggregate Key | Сұрыптау кілтін және агрегаттау тобын анықтау; Value бағандары жарияланған агрегаттау функцияларымен бірігеді |

`user_count` мәнін кәдімгі `SUM` ретінде анықтамаңыз: бір пайдаланушы бірнеше кіріс топтамасында кездесуі мүмкін болғандықтан, бірегей пайдаланушылар қосылмайды. Модельге біріктірілетін бірегей сан қажет болса, Bitmap немесе HLL күйін пайдаланыңыз.

Aggregate Key бағандары сұрыптау кілтін де құрайды. Бұл курс сынақ ортасында бір BE болғандықтан және алынған жиынтық шағын болғандықтан, кесте бір Bucket бар `HASH(region)` таратуды пайдаланады. Оның айқын Partition бөлімі жоқ, өйткені бұл шағын жиынтықта дербес Partition деңгейіндегі өмірлік цикл талабы жоқ. Бұл таңдаулар Partition, Bucket және сұрыптау кілті әмбебап үлгіге емес, дерек көлемі, сүзу, тарату және өмірлік цикл талаптарына сай болуы тиіс екенін көрсетеді.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS daily_event_metrics (
    event_date DATE NOT NULL,
    region VARCHAR(16) NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    event_count BIGINT SUM NOT NULL DEFAULT "0",
    total_revenue DECIMAL(18,2) SUM NOT NULL DEFAULT "0.00"
)
AGGREGATE KEY(event_date, region, event_type)
DISTRIBUTED BY HASH(region) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE daily_event_metrics")

lab.insert("""
INSERT INTO daily_event_metrics (
    event_date, region, event_type, event_count, total_revenue
)
SELECT
    TO_DATE(event_time),
    region,
    event_type,
    COUNT(*),
    SUM(revenue)
FROM events_modelled
GROUP BY TO_DATE(event_time), region, event_type
""", title="Build the daily reporting grain")

lab.sql("""
SELECT
    'events_modelled' AS table_name,
    COUNT(*) AS stored_rows,
    COUNT(*) AS represented_event_rows,
    SUM(revenue) AS represented_revenue
FROM events_modelled
UNION ALL
SELECT
    'daily_event_metrics',
    COUNT(*),
    SUM(event_count),
    SUM(total_revenue)
FROM daily_event_metrics
ORDER BY table_name
""", title="Detail and summary model reconciliation");


<a id="unit-u.1103ae801d0246a99b08eec033fe922b"></a>

**Күтілетін нәтиже**

Нәтиже түйіршіктілік өзгерісін көрсете отырып, 4-бөлімдегі салыстырып тексеруді қайталайды:

| table_name | stored_rows | represented_event_rows | represented_revenue |
|---|---:|---:|---:|
| daily_event_metrics | 280 | 10,158,080 | 39,984,455.64 |
| events_modelled | 10,158,080 | 10,158,080 | 39,984,455.64 |

`events_modelled` әр бастапқы оқиғаға бір жол сақтайды, сондықтан сақталған және көрсетілетін жол сандары бірдей. `daily_event_metrics` тек 280 күн-аймақ-оқиға түрі тіркесін сақтайды, бірақ оның қосылатын метрикалары барлық 10,158,080 оқиғаны және сол жалпы түсімді бәрібір көрсетеді. Кішірек `stored_rows` мәні жиынтық түйіршіктілігін растайды; көрсетілетін жиынтықтардың сәйкес келуі агрегаттаудың жарияланған метрикаларды сақтағанын тексереді.

Жиынтық `event_date`, `region`, `event_type`, `event_count` және `total_revenue` арқылы жауап берілетін сұрақтарды қолдайды. Жеке оқиғаны, пайдаланушыны, өнімді немесе бірегей пайдаланушы санын қалпына келтіре алмайды, өйткені ол өрістер түйіршіктілігінен тыс. Толық оқиғалар мен күндік метрикаларды бөлек кестелерде сақтау әртүрлі бизнес нысандарын білдіретін жолдарды араластыруға жол бермейді.

<a id="unit-u.747e1ffd5a294749afc80add16cb6060"></a>

<a id="6-practice-reconcile-one-day-of-the-summary-with-the-detail"></a>

## 6. Жаттығу: жиынтықтың бір күнін егжей-тегжейлі деректермен салыстырып тексеру

Бір күнді, 2020-03-01, таңдап, ештеңе орындамас бұрын болжаңыз:

- түйіршіктілігі бір `(event_date, region, event_type)` тобы екенін және сол күні тек `cart` пен `purchase` оқиғалары барын ескеріп, `daily_event_metrics` сол күнге қанша жол сақтайтынын;
- сол күн үшін `daily_event_metrics` және `events_modelled` арасындағы қай екі қосынды сәйкес болуы керектігін.

Содан кейін әр кесте үшін 2020-03-01 күнінің сақталған жолдарын, көрсетілетін оқиғаларын және түсімін бір нәтижеде қайтаратын бір сұрау жазыңыз.

In [ ]:
# Your code here

<a id="unit-u.707d3296e9024bf0a6230d422f919edb"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
from doris_course.ui import card

one_day = lab.sql("""
SELECT 'daily_event_metrics' AS source,
       COUNT(*) AS stored_rows,
       SUM(event_count) AS represented_events,
       SUM(total_revenue) AS revenue
FROM daily_event_metrics
WHERE event_date = '2020-03-01'
UNION ALL
SELECT 'events_modelled', COUNT(*), COUNT(*), SUM(revenue)
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
ORDER BY source
""", title="2020-03-01 in both models")

assert one_day["represented_events"].astype(int).nunique() == 1
assert one_day["revenue"].astype(str).nunique() == 1
card("Both models represent the same events and revenue for 2020-03-01.", "ok", "Practice checked");

<a id="unit-u.defe0ecb4e9c4140a9d46ef9f4ed8087"></a>

**Күтілетін нәтиже**

| source | stored_rows | represented_events | revenue |
|---|---:|---:|---:|
| daily_event_metrics | 16 | 75,259 | 5,670,241.29 |
| events_modelled | 75,259 | 75,259 | 5,670,241.29 |

Жасыл белгі көрсетілетін екі сан мен түсімнің сәйкес екенін растайды. 2020-03-01 күнінде `view` оқиғалары болмағандықтан, жиынтық сол күнге 16 жол, 8 аймақ × 2 оқиға түрін сақтайды. `COUNT(*)` әр кестенің түйіршіктілігіндегі жолдарды санайды, сондықтан 16 және 75,259 сандары әдейі әртүрлі; салыстырып тексеру екі түйіршіктілік те көрсете алатын метрикаларды — `SUM(event_count)` және түсім жиынтықтарын пайдаланады.

<a id="unit-u.3522b7581f4c424b9cc2d946d1f55e5c"></a>

<a id="stop-the-doris-sandbox"></a>

### Doris сынақ ортасын тоқтату

CPU мен жадты босату үшін осы қосымша ұяшықты орындаңыз. Ол Doris тоқтатады, бірақ контейнерді, образды, атаулы томдарды және 4-зертханалық жұмыс кестелерін сақтайды.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta
docker inspect --format 'container={{.State.Status}}' doris-rta
""", title="Stop the Doris sandbox");


<a id="unit-u.12d95d3cb1974f0d9f13d45d49a386aa"></a>

**Күтілетін нәтиже:** Docker `container=exited` деп хабарлайды. 1-деңгейдің бастапқы жиыны мен 4-зертханалық жұмыс кестелері атаулы томдарда қалады.

<a id="restart-the-doris-sandbox"></a>

### Doris сынақ ортасын қайта іске қосу

Сол ортамен жалғастыру үшін осы ұяшықты орындаңыз. Ол бар контейнерді іске қосып, FE түйініне қайта қосылады және сақталған 4-зертханалық жұмыс нәтижелерін тексереді.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашылады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.


In [ ]:
lab.start_container()

lab.connect()
lab.execute("USE doris_course")

lab.sql("""
SELECT
    (SELECT COUNT(*) FROM events_modelled) AS modelled_event_rows,
    (SELECT SUM(event_count) FROM daily_event_metrics) AS represented_event_rows,
    (SELECT SUM(total_revenue) FROM daily_event_metrics) AS represented_revenue
""", title="Recovered Lab 4 models");


<a id="unit-u.6591c566b43440bc9a619c532823d609"></a>

**Күтілетін нәтиже:** контейнер `healthy` күйіне оралады. Нәтиже 10,158,080 модельденген оқиға жолын, 10,158,080 көрсетілетін оқиға жолын және 39,984,455.64 көрсетілетін түсімді хабарлайды.

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

Дереккөз шартын толық, сұрауға дайын оқиға моделіне айналдырып, содан кейін есеп беру талабын күндік жиынтық түйіршіктілігіне айналдырдыңыз. Кесте модельдері, Key бағандары, қосылатын метрикалар, NULL рұқсаты, әдепкі мәндер, тарату, Bucket саны және Partition таңдаулары сол талаптардан шықты. Бір BE орналасуын өнімділік сынағы деп қабылдамай, жасалған екі айлық Partition мен олардың 20 Tablet нысанын тексердіңіз. Соңында түйіршіктілікке үйлесімді бизнес сұрағы үшін жиынтықты егжей-тегжейлі дереккөзімен салыстырып тексердіңіз.

Ресми анықтамалар: [Кесте модельдеріне шолу](https://doris.apache.org/docs/4.x/table-design/data-model/intro/) · [Кесте модельдерінің үздік тәжірибелері](https://doris.apache.org/docs/4.x/table-design/data-model/tips/) · [CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/) · [CAST және TRY_CAST](https://doris.apache.org/docs/4.x/sql-manual/basic-element/sql-data-types/conversion/cast-expr/) · [Aggregate Key моделі](https://doris.apache.org/docs/4.x/table-design/data-model/aggregate/)

<a id="unit-u.9625aa2748a24e34985b47a46238dea3"></a>

<a id="next"></a>

## Келесі қадам

[4-тест: Apache Doris жүйесінде деректерді модельдеу](quiz04_schema_and_modeling_choices.ipynb) ашыңыз, содан кейін [5-модуль: Apache Doris жүйесінде деректерді талдау](../module05-analyzing/course.md) бөліміне өтіңіз.